# TorchVision

## Transfer Learning

Ao treinar um modelo de rede neural, dificilmente configuramos toda a rede do zero. Na maioria dos casos, utilizamos um modelo consolidado pré-treinado como ResNet ou MobileNet e apenas o ajustamos para resolver nosso problema específico. Esses modelos foram treinados com milhões de imagens para reconhecer centenas de categorias. Portanto, uma coisa que eles já conseguem fazer muito bem é extrair sinais valiosos das imagens.

### Estrutura Universal

Os modelos de rede neural têm uma estrutura muito bem definida dividida em três partes principais: Pé (Stem); Corpo (Backbone); Cabeça (Head).

O Stem da rede neural é a porta de entrada que recebe os pixels brutos da imagem. Nesse estágio, nos preocupamos com a eficiência de processamento. Portanto, é aplicada a primeira camada de convolução que mapeia os três canais RGB para os primeiros canais da rede, diminuindo a resolução da imagem para não exaurir a memória do sistema ou a memória de vídeo. 

O Backbone é todo o corpo da rede neural que envolve camadas e camadas de convoluções que têm como função reconhecer linhas, cantos, ângulos, formatos, texturas e padrões visuais complexos. É no backbone que reside a capacidade da rede neural de aprender e ocorre o aumento do número de canais em detrimento da resolução da imagem, aumentando a capacidade da rede de reconhecer padrões nas imagens.

Por fim, temos a ponte (gap) e a cabeça da rede neural. A ponte é aquela peça única que colapsa o mapa 2D em um vetor unidimensional. Já a cabeça é a peça final que conecta os atributos aos logits e é responsável pela classificação.

### Como Modelos Pré-treinados Nomeiam as Partes

Junto à lista de modelos pré-treinados, incluímos a rede neural que configuramos anteriormente para explicitar a importância de definir bem a classe de definição. Até mesmo a nossa própria rede neural pode ser manipulada a fim de resolver outros problemas.

| Família | Backbone | Gap | Head |
| ------- | -------- | --- | ---- |
| ResNet  | model.conv1, model.layer1 - model.layer4 | model.avgpool | model.fc |
| MobileNet  | model.features | model.avgpool | model.classifier |
| EfficientNet  | model.features | model.avgpool | model.classifier |
| SimpleCNN  | model.features | model.gap | model.classifier |

### As duas Fases do Transfer Learning

#### Fase 1

1. Congelamento de todos os gradientes da rede neural
2. Substituição da cabeça da rede neural
3. Treinamento parcial da cabeça

Na primeira fase, começamos congelando todos os gradientes da rede neural. Isso tornará a rede completamente incapacitada de aprender. Os pesos da rede pré-treinada desde o pé até a cabeça da rede neural não serão perdidos ou modificados de qualquer forma.

O segundo passo é substituir a cabeça da rede. Redes como a ResNet, MobileNet e etc. São treinadas para reconhecer padrões que possibilitam classificar as imagens em centenas ou milhares de classes diferentes. No nosso problema, precisamos classificar em apenas três classes. Por isso, substituímos a cabeça original da rede por uma cabeça que entregue apenas três logits. Agora, a única parte da rede neural que não está com os gradientes congelados é a cabeça.

O último passo é realizar um treinamento da rede neural. Ela processará as imagens, calculará os logits, extrairá a perda, realizará a backpropagation e ajustará os pesos normalmente. Exceto por um detalhe. A única parte da rede neural que terá os pesos ajustados nesse momento é a cabeça, devido ao congelamento dos gradientes. 

Nessa primeira fase, estamos aproveitando a capacidade geral que a rede neural tem de reconhecer objetos e padrões visuais para treinar a nova cabeça, a fim de que ela refine os próprios pesos.

#### Fase 2

1. Destravamento da última camada convolucional do backbone
2. Diminuição da capacidade de aprendizado da última camada convolucional
3. Treinamento geral da rede neural

Na segunda fase, descongelamos a última camada convolucional do backbone da rede neural. Com isso, esta camada agora poderá ser ajustada para o nosso problema específico. Mas por quê congelamos essa parte antes de treinar a cabeça? Bom, como sabemos, inicialmente os pesos da rede neural são completamente aleatórios. Portanto, os pesos que a cabeça que definimos inicialmente tem são fezes puríssimas. Ao tentar treinar a rede neural sem congelar o backbone, o treinamento destruiria completamente os pesos que a rede neural tinha aprendido ao processar milhões de imagens. 

Ao treinar a rede neural modificando os pesos apenas da cabeça, teremos uma rede neural com um bom backbone capaz de detectar padrões gerais de classificação e uma cabeça capaz de utilizar esses padrões gerais para classificar as imagens para o nosso problema específico.

Ainda podemos fazer melhor, ajustando levemente os pesos da última camada convolucional do backbone. Para isso, definimos um learning rate menor que o padrão apenas para essa camada. Uma velocidade de 1e-5 (0.00001) ao invés de 1e-3 (0.001), por exemplo. Já a cabeça deve continuar com o padrão de 1e-3 (0.001).

Por fim, treinamos a rede neural novamente por mais algumas épocas. Nesse segundo treinamento, serão ajustados os pesos apenas para a última camada convolucional do backbone e para a cabeça da rede neural. Devido ao learning rate diminuído da última camada convolucional do backbone, essa camada sofrerá ajustes minúsculos durante o treinamento, refinando ela para o nosso problema específico. Agora, ao invés de identificar padrões gerais como focinhos de cachorro e rodas de bicicleta, ela será capaz de identificar padrões específicos do nosso problema. E a cabeça continuará aprendendo em velocidade normal para conseguir se ajustar à essa mudança de leitura do backbone. Ao final deste treinamento, teremos um backbone que é capaz de reconhecer os padrões específicos do problema e uma cabeça que é capaz de utilizar esses padrões para classificar as imagens.

### Exemplo Prático

Iniciando a prática, definimos os pesos do modelo que o próprio torchvision disponibiliza, instanciamos o avaliador e o modelo de nossa preferência. Nesse caso, o ResNet50. Para instanciá-lo, utilizamos o método resnet50 de torchvision.models. Este método recebe um parâmetro weights que deve receber os pesos. Por padrão, passamos os pesos do torchvision.

In [ ]:
from torch import nn
from torchvision import models
from torchvision.models import ResNet

from deep_learning.config import MODELS_DIR

weights = models.ResNet50_Weights.DEFAULT
model: ResNet = models.resnet50(weights=weights)
criterion = nn.CrossEntropyLoss()
save_path = MODELS_DIR / "model.pth"

O primeiro passo é congelar os gradientes da rede neural. Fazemos isso iterando os parâmetros do modelo e definindo a propriedade requires_grad como False.

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

Depois, substituímos a cabeça do modelo. Abaixo, atribuímos um nn.Sequential à propriedade fc do modelo ResNet. Para modelos como MobileNet, a propriedade que referencia a cabeça da rede tem um nome diferente. No código abaixo, recuperamos a quantidade de canais de input da cabeça original do modelo e utilizamos ela na nova cabeça para reduzir gradualmente os canais até classificar a imagem em três classes.

In [ ]:
original_in_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(in_features=original_in_features, out_features=128),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(in_features=128, out_features=3),
)

Agora, temos de executar a primeira fase de treinamento. Para isso, transferimos o modelo para o dispositivo em uso. Definimos um otimizador para todos os parâmetros da rede, aplicando a velocidade de aprendizado referente à cabeça, definida no arquivo de configurações. Podemos reaproveitar a função train_model que tínhamos implementado antes. Mas agora, ela também pode receber os parâmetros epochs para definir o número de épocas e save_path para definir o diretório em que os pesos devem ser salvos. Como congelamos todos os parâmetro e substituímos a cabeça, nesse momento, o otimizador modificará apenas os parâmetros da cabeça, mesmo passando todos os parâmetros do modelo como parâmetro para torch.optim.Adam.

In [ ]:
import torch

from deep_learning.config import DEVICE, LEARNING_RATE_HEAD, NUM_EPOCHS_PHASE_1
from main import train_model

model.to(DEVICE)

optimizer_phase_1 = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE_HEAD)

train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer_phase_1,
    epochs=NUM_EPOCHS_PHASE_1,
    device=DEVICE,
)

O próximo passo é realizar o treinamento mais refinado. Para isso, destravamos os parâmetros da última camada do modelo pré-treinado, que pode ser acessada pela propriedade layer4 do modelo.

In [ ]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

Definimos um otimizador para a segunda fase de treinamento. O Adam pode receber uma lista de dicionários. Cada dicionário poderá ter uma propriedade params para indicar quais parâmetros devem ser ajustados e uma propriedade lr para definir o ritmo do ajuste. Na segunda fase, aplicamos uma velocidade de aprendizado menor (1e-5) que foi definida no config.py assim como a velocidade de aprendizado padrão (1e-3). Essencialmente, o que está acontecendo é que estamos determinando que os parâmetros da última camada do backbone aprenderão na velocidade 1e-5 e os parâmetros da cabeça continuarão aprendendo na velocidade 1e-3, assim como antes.

In [ ]:
from deep_learning.config import LEARNING_RATE_BACKBONE

optimizer_phase_2 = torch.optim.Adam(
    [
        {
            "params": model.layer4.parameters(),
            "lr": LEARNING_RATE_BACKBONE,
        },
        {"params": model.fc.parameters(), "lr": LEARNING_RATE_HEAD},
    ]
)

Para terminar, executamos novamente a função de treinamento. Dessa vez, substituindo o número de épocas e o otimizador para esta fase.

In [ ]:
from deep_learning.config import NUM_EPOCHS_PHASE_2

train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer_phase_2,
    epochs=NUM_EPOCHS_PHASE_2,
    device=DEVICE,
)

Geralmente, não é recomendado executar muitas épocas de treinamento na primeira fase. Deverão ser de três a cinco épocas, no máximo. O objetivo na primeira fase é apenas ajustar grosseiramente os parâmetros da cabeça da rede. Já a segunda fase poderá ter mais épocas. Nesse projeto, estamos definindo 5 épocas para a fase 1 e 10 épocas para a fase 2.

Quando definimos e treinamos nossa própria rede neural do zero, alcançamos uma acurácia de teste 89.33%. Ao aplicar transfer learning com o modelo ResNet50, alcançamos incríveis 98% de acurácia. Antes, o modelo errava 1 classificação a cada 10 imagens. Agora, o modelo erra 1 classificação a cada 50 imagens.

Para transfer learning com qualquer outro modelo, a diferença é muito singela. Apenas variando na forma de fazer coisas como trocar a cabeça da rede e desbloquear a última camada do backbone.

## Ingestão de Dados

O segundo problema que o TorchVision resolve é o carregamento dos dados em lotes. Quando definimos o dataset manualmente, liamos a imagem com PIL, convertíamos para numpy, fazendo transformações enormalizações, convertíamos para torch e finalmente a imagem podia ser usada. Esse processo cria cópias transitórias da imagem. Tínhamos a mesma imagem como um objeto do PIL, como um tensor numpy e como um tensor do torch. O TorchVision resolve esse problema lendo a imagem, decodificando via rotinas compiladas em C++ e escrevendo diretamente um tensor do pytorch.

### Definindo Conjuntos de Dados

Além do quesito performático, há também a questão do boilerplate. Antes de definirmos os conjuntos de dados com torchvision, lembre-se do conteúdo anterior do arquivo dataset.py.

In [ ]:
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from deep_learning.config import (
    BATCH_SIZE,
    DATA_DIR,
    TEST_DATA_DIR,
    TRAIN_DATA_DIR,
    VALIDATION_DATA_DIR,
)
from deep_learning.data.transforms import (
    test_transform,
    train_transform,
    validation_transform,
)


class ImageFolderDataset(Dataset):
    def __init__(self, root_dir: str | Path = DATA_DIR, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        self.classes = sorted(
            [dir.name for dir in self.root_dir.iterdir() if dir.is_dir()]
        )
        self.class_to_index = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []

        for cls_name in self.classes:
            folder = self.root_dir / cls_name

            for img_path in folder.glob("*.*"):
                if img_path.suffix.lower() in (".jpg", ".png", ".jpeg"):
                    self.samples.append((img_path, self.class_to_index[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index: int):
        img_path, label = self.samples[index]

        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label
        

train_dataset = ImageFolderDataset(root_dir=TRAIN_DATA_DIR, transform=train_transform)
test_dataset = ImageFolderDataset(root_dir=TEST_DATA_DIR, transform=test_transform)
validation_dataset = ImageFolderDataset(root_dir=VALIDATION_DATA_DIR, transform=validation_transform)

train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

validation_data_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

Agora, ao invés de termos de definir uma classe de dataset para as pastas, podemos simplesmente utilizar uma classe ImageFolder de torchvision.datasets. Essa classe receberá como parâmetro root o caminho até a pasta que diz respeito ao dataset e poderá receber como parâmetro transform uma função que transforma as imagens consumidas. Essa classe nativa do torchvision já faz exatamente o mesmo trabalho que antes fizemos manualmente. Eliminando código boilerplate pesado. 

In [ ]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(root=TRAIN_DATA_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(root=TEST_DATA_DIR, transform=test_transform)
validation_dataset = datasets.ImageFolder(
    root=VALIDATION_DATA_DIR, transform=validation_transform
)

A definição dos data loaders não mudará em absolutamente nada. Pois o dataset definido pelo torchvision funciona exatamente da mesma forma.

In [ ]:
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

# ...

Outros dois parâmetros que podemos passar para a classe DataLoader são os parâmetros persistent_workers e prefetch_factor. O persistent_workers mantém os processos de ingestão de dados vivos entre épocas, eliminando a necessidade de destruir os processos e depois reconstruí-los novamente. Já o parâmetro prefetch_factor definirá um número de lotes que devem ser adiantados. No projeto, estamos adiantando dois lotes. Dessa forma, quando a função de treinamento pede o próximo lote, já teremos ao menos o próprio lote e os dois próximos carregados. Dessa forma, a GPU nunca fica esperando pelo carregamento de novos lotes.

In [ ]:
# ...

import os

train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=os.cpu_count() or 4,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
    prefetch_factor=2,
)

# ...

### Utilizando Datasets do TorchVision

O TorchVision disponibiliza alguns datasets de exemplo para podermos prototipar e experimentar uma rede neural antes de precisarmos procurar, avaliar e baixar um enorme volume de dados para resolver um determinado problema. Um dos datasets é o FashionMNIST. Para importar um dataset, devemos importar a classe dele de torchvision.datasets. Depois, basta instanciá-lo, passando como parâmetros para o construtor da classe um parâmetro root que indica em que diretório o dataset deve ser carregado. Um parâmetro train que define se queremos baixar o conjunto de treino do dataset ou o conjunto de validação. Um parâmetro download que define se queremos transferir os dados do dataset para o disco na localização indicada ou não. Poderemos utilizar o dataset normalmente se não o transferirmos para o disco. A classe do dataset pode também receber um parâmetro transform que é uma função que transformará as imagens de acordo com a nossa necessidade.

In [ ]:
from torchvision.datasets import FashionMNIST

from deep_learning.config import DATA_DIR

fashion_mnist = FashionMNIST(
    root=DATA_DIR / "fashion_mnist", train=True, download=True
)


Para utilizar o dataset, podemos criar um data loader para ele. Basta passar o objeto do dataset como parâmetro dataset para a classe DataLoader. Funcionará exatamente como se as imagens estivessem no disco.

In [ ]:
from torch.utils.data import DataLoader

data_loader = DataLoader(
    dataset=fashion_mnist,
    # ...
)

### Transformações

Até o momento, estamos transformando as imagens manualmente ao carregá-las com PIL. Modificando sequencialmente as dimensões, rotação, transposição e normalização. Esses processos todos rodam no processador, uma imagem de cada vez. Além de faltar em um outro tipo de normalização centralizada em zero. Esse tipo de normalização é exigido para que os filtros de redes pré-treinadas funcionem com o máximo de acurácia. Os pesos de fábrica dos modelos pré-treinados são calculados a partir de valores de pixels centralizados ao redor do zero. Por isso, para que as primeiras camadas de convolução dos modelos consigam extrair o máximo de sinais, temos de normalizar os pixels exatamente dessa forma.

No arquivo transforms.py, onde guardamos as funções transformadoras, vamos passar a utilizar os transformadores do torchvision. Para isso, importamos um objeto v2 de torchvision.transforms, que encapsula grande parte dos recursos que precisaremos. Para definir uma sequência de operações, utilizamos a classe Compose que receberá como primeiro parâmetro uma lista com os passos em ordem. A classe ToImage garante que a imagem seja convertida em um tensor nativo. RandomResizedCrop irá modificar a escala da imagem e cortá-la nas dimensões desejadas. O parâmetro scale recebe uma tupla na qual o primeiro item é a escala mínima e o segundo item é a escala máxima. O parâmetro size define as dimensões que a imagem deverá ver e antialias define se o mecanismo deve evitar o serrilhado e artefatos visuais falsos. RandomHorizontalFlip aplicará transposição de acordo com uma propabilidade p. RandomRotation aplicará uma rotação aleatória de acordo com um intervalo de graus definido em degrees, variando do valor absoluto negativo ao positivo. ColorJitter modificará a luminosidade, contraste e saturação da imagem até um limite definido. ToDtype modificará o tipo dos dados do tensor. O parâmetro scale de ToDtype define se a divisão por 255 deve ser realizada automaticamente ou não. Por fim, o Normalize normalizará os valores dos pixels entre -1 e 1 de acordo com as médias e desvio padrão do dataset. Os valores utilizados abaixo são padrão da ResNet.

In [ ]:
import torch
from torchvision.transforms import v2

from deep_learning.config import IMAGE_SIZE

train_transform = v2.Compose(
    [
        v2.ToImage(),
        v2.RandomResizedCrop(size=IMAGE_SIZE, scale=(0.7, 1.0), antialias=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomRotation(degrees=15),
        v2.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]
)

# ...

Existem outras classes transformadoras como a Resize, que redimensiona a imagem, mantendo as proporções. Você não precisa se lembrar de todas as classes transformadoras que existem e todos os parâmetros que elas recebem. Precisa se lembrar apenas de que é possível definir fluxos de transformação com diversas funções de forma facilitada.

In [ ]:
# ...

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(size=IMAGE_SIZE, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ...

Diferentes redes neurais pré-treinadas processam as imagens antes do treinamento, validação e teste de formas diferentes. Por exemplo, um modelo pode redimensionar as imagens para 224x224, outro pode redimensionar para 256x256, um último poderia redimensionar para 288x288. Não é muito recomendável ter de se lembrar dessas diferenças. Por isso, nos objetos de pesos dos modelos, temos uma forma de acessar as funções transformadoras que a rede neural utilizava quando foi treinada com aquelas centenas de milhares de imagens. 

No objeto dos pesos padrão, há um método transforms que retorna a função transformadora padrão. Podemos simplesmente utilizar eles como transformadores para os estágios de validação e teste.

In [ ]:
# ...

test_transform = models.ResNet50_Weights.DEFAULT.transforms()

validation_transform = models.ResNet50_Weights.DEFAULT.transforms()

## Interpretabilidade